# Term Frequency–Inverse Document Frequency

TF-IDF (Term Frequency–Inverse Document Frequency) is a statistical method used in natural language processing and information retrieval to evaluate how important a word is to a document in relation to a larger collection of documents.

The intuition: a word matters to a document when it is **frequent inside that document** (TF) but **rare across the whole collection** (IDF). A word like *"the"* is frequent everywhere, so it carries no signal; a word like *"photosynthesis"* is rare, so when it does appear it is highly distinctive. TF-IDF multiplies these two signals together.

TF-IDF combines two components:

### 1. Term Frequency (TF)

Measures how often a word appears in a single document, normalized by the document's length so that long and short documents are comparable. A higher value means the term is more central to *that* document's content.

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$

where $f_{t,d}$ is the number of times term $t$ appears in document $d$, and the denominator is the total number of terms in $d$. The result lies in $[0, 1]$.

### 2. Inverse Document Frequency (IDF)

Measures how rare a term is across the **entire corpus**. It down-weights words that appear in many documents (common, low-information) and up-weights words that appear in few (rare, high-information). Where TF looks at one document, IDF looks at all of them.

The textbook definition is:

$$\text{IDF}(t, D) = \log \frac{N}{n_t}$$

where $N$ is the total number of documents in the corpus $D$, and $n_t = |\{d \in D : t \in d\}|$ is the **document frequency** — the number of documents that contain term $t$ (counted once per document, regardless of how many times the term occurs inside it).

**The problem with this raw form.** When a term appears in *no* document, $n_t = 0$ and $\log(N/0)$ is undefined — a division by zero. This is not exotic: any query for a word the corpus has never seen (out-of-vocabulary) triggers it. The raw form is also unstable for terms in every document ($\log(N/N) = 0$), which collapses to the same value you would naively return for an absent term.

**Smoothed IDF (the form used in this notebook).** To stay finite and well-behaved we add 1 to both the numerator and denominator, and add 1 to the whole result — the same smoothing scikit-learn's `TfidfVectorizer` uses:

$$\text{IDF}(t, D) = \log \frac{1 + N}{1 + n_t} + 1$$

What the smoothing buys us:

- **No division by zero.** The $1 + n_t$ denominator is never 0, so absent terms ($n_t = 0$) no longer crash — they simply get the highest IDF, which is the correct behavior since the rarest possible term should be the most distinctive.
- **Never negative.** The trailing $+ 1$ guarantees $\text{IDF} \geq 1$ even for a term that appears in every document. Without it, a very common term could get a negative weight and *subtract* from a document's relevance — usually undesirable.
- **Never zero.** Every term keeps a non-zero weight, so a term that is genuinely present is never completely erased from the score.

Effectively, the smoothing behaves *as if* one extra document containing every term were added to the corpus — a gentle regularizer that keeps the math defined at the edges without changing the ranking of normal terms.

### Combining the two

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

A term scores high only when **both** factors are high: it must be frequent in the document *and* rare in the corpus.

---

This balance allows TF-IDF to highlight terms that are both frequent within a specific document and distinctive across the collection, making it a useful tool for tasks like search ranking, text classification and keyword extraction.


In [73]:
import re
import numpy as np

## Term Frequency

We tokenize both the term and the document with a single shared tokenizer (`tokenize`), then divide the count of the term's token by the total number of tokens in the document. Using **one** tokenizer for the numerator and the denominator guarantees they always agree on what a "word" is — so the result can never exceed 1, and punctuation/casing are handled consistently. An empty document returns `0.0` rather than dividing by zero.


In [68]:
TOKEN = r'\w+(?:\.\w+)*'

def tokenize(text):
    return re.findall(TOKEN, text.lower())

def eval_term_frequency(term, doc):
    tokens = tokenize(doc)
    if not tokens:
        return 0.0
    target = tokenize(term)
    return tokens.count(target[0]) / len(tokens) if target else 0.0

In [69]:
# --- Unit tests for eval_term_frequency ---
# Make the CODE pass these. Don't change the tests to fit the code.
# Run this cell after each edit to your function and watch which ones flip green.

import math

def _approx(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=0, abs_tol=tol)

def run_tf_tests():
    cases = [
        # (term, doc, expected, note)
        ("cat", "the cat sat",               1/3, "basic: 1 of 3 words"),
        ("the", "the cat sat the the",       3/5, "repeated term"),
        ("dog", "the cat sat",               0.0, "term absent -> 0"),
        ("cat", "the cat scattered cats",    1/4, "must NOT match substrings in 'scattered'/'cats'"),
        ("Cat", "cat CAT cat",               1.0, "case-insensitive: 'Cat' matches cat/CAT/cat -> 3/3"),
        ("CAT", "The Cat sat",               1/3, "case-insensitive: term and doc differ in case"),
        ("cat", "cat,cat cat",               1.0, "punctuation: numerator & denominator must agree on tokens -> 3 cats / 3 tokens"),
        ("c++", "i love c++ and c++",        2/5, "term is DATA not regex: 'c++' must not crash or mis-match"),
        ("u.s.a", "u.s.a is big",            1/3, "dots in term must be literal, not regex 'any char'"),
    ]

    passed = 0
    for i, (term, doc, expected, note) in enumerate(cases, 1):
        try:
            got = eval_term_frequency(term, doc)
            ok = _approx(got, expected)
            status = "PASS" if ok else "FAIL"
            passed += ok
            print(f"[{status}] test {i}: tf({term!r}, {doc!r}) = {got}  (expected {expected})  # {note}")
        except Exception as e:
            print(f"[ERROR] test {i}: tf({term!r}, {doc!r}) raised {type(e).__name__}: {e}  # {note}")

    # Edge case: empty document should not crash. Decide your own contract
    # (return 0.0? raise a clear error?) and assert it here once you've decided.
    try:
        got = eval_term_frequency("cat", "")
        print(f"[INFO ] empty-doc: tf('cat', '') = {got}  (you decide: is this the contract you want?)")
    except Exception as e:
        print(f"[INFO ] empty-doc: raised {type(e).__name__}: {e}  (intentional? document your choice)")

    print(f"\n{passed}/{len(cases)} core tests passing")

run_tf_tests()


[PASS] test 1: tf('cat', 'the cat sat') = 0.3333333333333333  (expected 0.3333333333333333)  # basic: 1 of 3 words
[PASS] test 2: tf('the', 'the cat sat the the') = 0.6  (expected 0.6)  # repeated term
[PASS] test 3: tf('dog', 'the cat sat') = 0.0  (expected 0.0)  # term absent -> 0
[PASS] test 4: tf('cat', 'the cat scattered cats') = 0.25  (expected 0.25)  # must NOT match substrings in 'scattered'/'cats'
[PASS] test 5: tf('Cat', 'cat CAT cat') = 1.0  (expected 1.0)  # case-insensitive: 'Cat' matches cat/CAT/cat -> 3/3
[PASS] test 6: tf('CAT', 'The Cat sat') = 0.3333333333333333  (expected 0.3333333333333333)  # case-insensitive: term and doc differ in case
[PASS] test 7: tf('cat', 'cat,cat cat') = 1.0  (expected 1.0)  # punctuation: numerator & denominator must agree on tokens -> 3 cats / 3 tokens
[PASS] test 8: tf('c++', 'i love c++ and c++') = 0.4  (expected 0.4)  # term is DATA not regex: 'c++' must not crash or mis-match
[PASS] test 9: tf('u.s.a', 'u.s.a is big') = 0.333333333333

## Inverse Document Frequency

We count the **document frequency** $n_t$ — how many documents in the corpus contain the term at least once (membership, not occurrence count, so a term appearing five times in one document still counts as one). We then apply the smoothed formula $\log\frac{1+N}{1+n_t} + 1$.

The smoothing (the two `+1`s in the fraction, plus the trailing `+1`) is what keeps this robust: an out-of-vocabulary term ($n_t = 0$) no longer divides by zero and instead receives the highest weight, and no term ever gets a negative or zero IDF. See the intro section for the full reasoning. Degenerate inputs — an empty corpus or an empty term — return `0.0` by contract (no corpus or no term means no signal).


In [116]:
def eval_inverse_document_frequency(term, corpus):
    N = len(corpus)
    t = tokenize(term)
    if N == 0:
        return 0
    if len(t) == 0:
        return 0
    n = 0
    for doc in corpus:
        d = tokenize(doc)
        if t[0] in d:
            n += 1
    return np.log((N+1)/(n+1)) + 1


In [117]:
# --- Unit tests for eval_inverse_document_frequency ---
# Make the CODE pass these. Don't change the tests to fit the code.
# CONTRACT: scikit-learn smoothing -> IDF(t) = log((1 + N) / (1 + df)) + 1
#   df(t) = number of DOCUMENTS containing t (>= 1 occurrence).
#   Never negative, never zero, never divides by zero.

import math

def _approx(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=0, abs_tol=tol)

def run_idf_tests():
    C = ["the cat sat", "the dog ran", "the bird flew"]   # N = 3

    cases = [
        # (term, corpus, expected, note)   expected = log((1+N)/(1+df)) + 1
        ("the", C,                                 math.log(4/4) + 1, "in ALL docs (df=3): log(4/4)+1 = 1.0"),
        ("cat", C,                                 math.log(4/2) + 1, "in 1 of 3 docs (df=1)"),
        ("dog", C,                                 math.log(4/2) + 1, "in 1 of 3 docs (df=1)"),
        ("cat", ["cat", "cat cat"],                math.log(3/3) + 1, "df counts DOCUMENTS not occurrences: df=2, N=2"),
        ("dog", ["the dog ran", "dogs everywhere", "no canine"], math.log(4/2) + 1, "'dogs' is a DIFFERENT token from 'dog' -> df=1"),
        ("THE", ["The cat", "the dog", "nothing"], math.log(4/3) + 1, "case-insensitive across docs -> df=2"),
        # edge cases are now real assertions, since the contract defines them:
        ("fish", C,                                math.log(4/1) + 1, "absent term (df=0): rarest -> HIGHEST idf, no crash"),
    ]

    passed = 0
    for i, (term, corpus, expected, note) in enumerate(cases, 1):
        try:
            got = eval_inverse_document_frequency(term, corpus)
            ok = _approx(got, expected)
            status = "PASS" if ok else "FAIL"
            passed += ok
            print(f"[{status}] test {i}: idf({term!r}, <{len(corpus)} docs>) = {got}  (expected {expected})  # {note}")
        except Exception as e:
            print(f"[ERROR] test {i}: idf({term!r}, ...) raised {type(e).__name__}: {e}  # {note}")

    # Degenerate input: empty corpus (N=0). Contract choice: return 0.0 (no corpus -> no signal).
    try:
        got = eval_inverse_document_frequency("cat", [])
        ok = _approx(got, 0.0)
        print(f"[{'PASS' if ok else 'FAIL'}] empty-corpus: idf('cat', []) = {got}  (expected 0.0 by contract)")
        passed += ok
    except Exception as e:
        print(f"[ERROR] empty-corpus: raised {type(e).__name__}: {e}")

    total = len(cases) + 1
    print(f"\n{passed}/{total} tests passing")

run_idf_tests()


[PASS] test 1: idf('the', <3 docs>) = 1.0  (expected 1.0)  # in ALL docs (df=3): log(4/4)+1 = 1.0
[PASS] test 2: idf('cat', <3 docs>) = 1.6931471805599454  (expected 1.6931471805599454)  # in 1 of 3 docs (df=1)
[PASS] test 3: idf('dog', <3 docs>) = 1.6931471805599454  (expected 1.6931471805599454)  # in 1 of 3 docs (df=1)
[PASS] test 4: idf('cat', <2 docs>) = 1.0  (expected 1.0)  # df counts DOCUMENTS not occurrences: df=2, N=2
[PASS] test 5: idf('dog', <3 docs>) = 1.6931471805599454  (expected 1.6931471805599454)  # 'dogs' is a DIFFERENT token from 'dog' -> df=1
[PASS] test 6: idf('THE', <3 docs>) = 1.2876820724517808  (expected 1.2876820724517808)  # case-insensitive across docs -> df=2
[PASS] test 7: idf('fish', <3 docs>) = 2.386294361119891  (expected 2.386294361119891)  # absent term (df=0): rarest -> HIGHEST idf, no crash
[PASS] empty-corpus: idf('cat', []) = 0  (expected 0.0 by contract)

8/8 tests passing
